# CAMELS: Time Series for the Website
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 22-04-2026<br>

**Introduction:**<br>
This script combines the daily discharge records with the daily basin meteorology computed from the EMO-1 dataset and exports a time series per gauging station to be plotted in the website.

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd

from ocab.config import Config
from ocab.plots.stations import plot_station_timeseries, create_station_html


## Configuration


In [ ]:
cfg = Config('config_CAMELS_v200.yml')

# paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'
path_out = Path('../../docs/timeseries/stations')
path_plots = path_out / 'plots'
path_plots.mkdir(exist_ok=True, parents=True)

# decimals in output timeseries
rounding = {
    'discharge_cms': 3,
    'discharge_mm': 1,
    'temp_degC': 1,
    'precip_mm': 1,
    'pet_mm': 1,
}

variables = {
    'ta_mean': 'temp_degC', 
    'pr_mean': 'precip_mm', 
    'e0_mean': 'pet_mm',
}



## Create time series


In [ ]:

# load stations
stations = gpd.read_file(cfg.path_gis / 'stations.geojson').set_index('id')

# process timeseries for each station
for ID in tqdm(stations.index, desc='stations'):

    ID = 1237 
    
    # discharge timeseries
    try:
        dis = pd.read_parquet(path_in / 'discharge' / f'{ID:04d}.parquet')
        dis.columns = ['discharge_cms']
        # compute specific discharge (mm/day)
        dis['discharge_mm'] = dis['discharge_cms'] / stations.loc[ID, 'catch_skm'] * 86400 / 1000
    except Exception as e:
        print(f'Error loading discharge timeseries for station {ID:04d}: {e}')
        continue

    # meteo timeseries
    try:
        ts = pd.read_parquet(path_in / 'meteo' / 'EMO1' / f'{ID:04d}.parquet').loc[ID]
        ts.rename(columns=variables, inplace=True, errors='ignore')
        # correct dates
        ts.index = ts.index.date - pd.Timedelta(days=1)
        ts.index.name = 'date'
        ts.index = pd.to_datetime(ts.index)
        start = max(ts.first_valid_index(), dis.first_valid_index())
        end = min(ts.last_valid_index(), dis.last_valid_index())
        ts = ts.loc[start:end]
    except Exception as e:
        print(f'Error loading meteo timeseries for station {ID:04d}: {e}')
        continue

    # merge timeseries
    ts = pd.concat([dis, ts], axis=1)
    ts = ts[ts.columns.intersection(rounding)].round(rounding)

    # # export timeseries
    # ts.to_parquet(path_out / f'{ID:04d}.parquet')

    # extract attributes and time series
    attrs = stations.loc[ID]

    # create time series plot
    try:
        title = '{0} - {1} - River {2} ({3})'.format(
            ID, 
            attrs['name'].title(), 
            attrs['river'].title(), 
            attrs['basin'].title()
        )
        fig = plot_station_timeseries(
            ts,
            area=attrs['catch_skm'],
            title=title,
            regime=attrs['regime'],
            save=True
        )

        # save plot as HTML
        create_station_html(
            fig, 
            path=path_plots / f'{ID}.html', 
            start=ts.index.min().strftime('%Y-%m-%d'), 
            end=ts.index.max().strftime('%Y-%m-%d')
        )
    except:
        print(f"The plot for time series {ID} couldn't be created")

stations:   0%|          | 0/1009 [00:00<?, ?it/s]